In [ ]:
import pandas as pd 
import sys
sys.path.append('..')  
df = pd.read_excel('../data/propostas_concatenadas.xlsx')

In [ ]:
df.columns

In [ ]:
df['Proposta Comercial'].value_counts()

Proposta repetida com maquinas diferentes

In [ ]:
# Mostrar casos onde Proposta + Local são iguais mas Máquinas são diferentes
df['proposta_local'] = df['Proposta Comercial'].astype(str) + '|' + df['Local'].astype(str)
proposta_local_counts = df['proposta_local'].value_counts()
duplicated_proposta_local = proposta_local_counts[proposta_local_counts > 1]

casos_diferentes = []
for combo in duplicated_proposta_local.index:
    group = df[df['proposta_local'] == combo]
    if group['Máquinas'].nunique() > 1:
        casos_diferentes.append(combo)

print(f"Casos com máquinas diferentes: {len(casos_diferentes)}")

if len(casos_diferentes) > 0:
    for combo in casos_diferentes:
        proposta, local = combo.split('|')
        group = df[df['proposta_local'] == combo]
        maquinas = list(group['Máquinas'].unique())
        print(f"\n{proposta} | {local} | Máquinas: {maquinas}")
        display(group[['Proposta Comercial', 'Máquinas', 'Local', 'Per', 'Nome do Cliente']].head())

df = df.drop('proposta_local', axis=1)  # Limpar coluna temporária

Juntar propostas repetidas + mesmo local com maquina diferente 

In [ ]:
# Juntar máquinas diferentes com mesma Proposta + Local no DataFrame original
df['combo_key'] = df['Proposta Comercial'].astype(str) + '|' + df['Local'].astype(str)
combo_counts = df['combo_key'].value_counts()
duplicated_combos = combo_counts[combo_counts > 1]

rows_to_remove = []

for combo in duplicated_combos.index:
    group = df[df['combo_key'] == combo]
    if group['Máquinas'].nunique() > 1:
        # Combinar máquinas e manter primeiro registro
        maquinas_combined = ', '.join(group['Máquinas'].dropna().unique().astype(str))
        base_idx = group.index.min()
        df.loc[base_idx, 'Máquinas'] = maquinas_combined
        rows_to_remove.extend(group.index[group.index != base_idx].tolist())

# Aplicar mudanças ao DataFrame original
df = df.drop(rows_to_remove).drop('combo_key', axis=1)

print(f"Linhas removidas: {len(rows_to_remove)} | Total final: {len(df)}")

# Mostrar exemplos consolidados
juntar = df[df['Máquinas'].astype(str).str.contains(',', na=False)]
if len(juntar) > 0:
    display(juntar[['Proposta Comercial', 'Máquinas', 'Local', 'Per', 'Nome do Cliente']].head())

Propostas repedidas com a mesma Maquina + Local

In [ ]:
# Mostrar duplicatas por Proposta + Máquinas + Local
df['combo'] = df['Proposta Comercial'].astype(str) + '|' + df['Máquinas'].astype(str) + '|' + df['Local'].astype(str)
duplicated_combos = df['combo'].value_counts()
duplicated_combos = duplicated_combos[duplicated_combos > 1]

print(f"Total de duplicatas completas: {len(duplicated_combos)}")

if len(duplicated_combos) > 0:
    for combo in duplicated_combos.index:
        proposta, maquinas, local = combo.split('|')
        duplicate_rows = df[df['combo'] == combo]
        print(f"\n{proposta} | {maquinas} | {local} ({duplicated_combos[combo]}x)")
        display(duplicate_rows[['Proposta Comercial', 'Máquinas', 'Local', 'Per', 'Nome do Cliente']].head())

df = df.drop('combo', axis=1)  # Limpar coluna temporária

In [ ]:
# Remover duplicatas (Proposta + Máquinas + Local) mantendo o mais recente
df['combo_full'] = df['Proposta Comercial'].astype(str) + '|' + df['Máquinas'].astype(str) + '|' + df['Local'].astype(str)

# Manter apenas o último (mais recente) de cada combinação
df_deduplicated = df.drop_duplicates(subset=['combo_full'], keep='last').drop('combo_full', axis=1)

removed_count = len(df) - len(df_deduplicated)
df = df_deduplicated

print(f"Duplicatas completas removidas: {removed_count} | Total final: {len(df)}")

In [ ]:
# Mostrar todas as propostas comerciais duplicadas (sem filtros)
duplicate_propostas = df['Proposta Comercial'].value_counts()
duplicate_propostas = duplicate_propostas[duplicate_propostas > 1]

print(f"Total de propostas comerciais duplicadas: {len(duplicate_propostas)}")

# Verificar se todos os locais são diferentes para cada proposta duplicada
propostas_locais_diferentes = 0
propostas_locais_iguais = 0

for proposta in duplicate_propostas.index:
    duplicate_rows = df[df['Proposta Comercial'] == proposta]
    unique_locals = duplicate_rows['Local'].nunique()
    total_rows = len(duplicate_rows)
    
    if unique_locals == total_rows:
        propostas_locais_diferentes += 1
    else:
        propostas_locais_iguais += 1

print(f"Propostas com todos os locais diferentes: {propostas_locais_diferentes}")
print(f"Propostas com locais repetidos: {propostas_locais_iguais}")

In [ ]:
# Modificar Proposta Comercial baseado no Local
# Se Local = JPN, adicionar _JPN ao final da Proposta Comercial
# Se Local = Muralha, manter igual

df['Proposta Comercial'] = df.apply(
    lambda row: row['Proposta Comercial'] + '_JPN' if row['Local'] == 'JPN' 
    else row['Proposta Comercial'], axis=1
)

In [ ]:
df.to_csv('../data/df_maquinas.csv', index=False)